# rosetta-bioc on Posit Cloud
## Differential expression with DESeq2 — entirely in Python

This notebook demonstrates the core rosetta-bioc value proposition:
write Python, get DESeq2 results back as a pandas DataFrame.

**Tested environment:** Python 3.12.11, R 4.6.1, Bioconductor 3.23, rosetta-bioc 0.3.2

### Prerequisites

Run these once in the Posit Cloud terminal before opening this notebook:

```bash
pip install rosetta-bioc matplotlib
```

```r
# In R console
BiocManager::install("DESeq2")
```

## 1. Load the airway dataset

`airway` is a standard Bioconductor RNA-seq dataset: 8 samples from human airway
smooth muscle cells, treated with dexamethasone (dex) or untreated (untrt).
We load it from R and pull the count matrix + metadata into Python.

In [ ]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

# Install airway if needed (only runs once)
ro.r("""
if (!requireNamespace("airway", quietly=TRUE))
    BiocManager::install("airway")
""")

# Load the dataset
ro.r("""
library(airway)
data(airway)
counts_r  <- assay(airway, "counts")
coldata_r <- as.data.frame(colData(airway))
""")

# Pull into Python
with (ro.default_converter + pandas2ri.converter).context():
    counts   = ro.conversion.rpy2py(ro.r["counts_r"])
    metadata = ro.conversion.rpy2py(ro.r["coldata_r"])

# Keep only the column we need for the design
metadata = metadata[["dex"]]
metadata["dex"] = metadata["dex"].astype(str)

# Align: counts columns must match metadata rows
counts = counts[metadata.index]

print(f"Count matrix : {counts.shape[0]:,} genes × {counts.shape[1]} samples")
print(f"Conditions   : {metadata['dex'].value_counts().to_dict()}")
counts.iloc[:4, :4]

## 2. Filter low-count genes

Genes with near-zero counts across all samples add noise and slow DESeq2 down.
Keep genes with at least 10 counts in at least 2 samples.

In [ ]:
keep = (counts >= 10).sum(axis=1) >= 2
counts_filt = counts.loc[keep]
print(f"After filtering: {counts_filt.shape[0]:,} genes retained ({keep.sum() / len(keep):.0%})")

## 3. Run DESeq2 via rosetta

One call — `quick_deseq2` constructs the DESeqDataSet, fits the model,
and returns results as a pandas DataFrame.

In [ ]:
import rosetta as rb

results = rb.quick_deseq2(
    counts_filt,
    metadata,
    design="~ dex",
    alpha=0.05,
)

print(type(results))  # RosettaDataFrame — a pandas DataFrame subclass
results.head()

## 4. Summary

In [ ]:
results.report()

## 5. Top differentially expressed genes

In [ ]:
sig = results.dropna(subset=["padj"]).query("padj < 0.05")
print(f"{len(sig):,} significant genes (FDR < 5%)")

top = sig.sort_values("log2FoldChange", key=abs, ascending=False).head(10)
top[["baseMean", "log2FoldChange", "lfcSE", "padj"]]

## 6. Volcano plot

In [ ]:
import matplotlib.pyplot as plt

plot_df = results.dropna(subset=["padj", "log2FoldChange"]).copy()
plot_df["neg_log10_padj"] = -np.log10(plot_df["padj"].clip(lower=1e-300))

is_sig = plot_df["padj"] < 0.05
is_up  = is_sig & (plot_df["log2FoldChange"] > 1)
is_dn  = is_sig & (plot_df["log2FoldChange"] < -1)

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(plot_df.loc[~is_sig, "log2FoldChange"],
           plot_df.loc[~is_sig, "neg_log10_padj"],
           s=6, color="#aaaaaa", alpha=0.5, label="not significant")
ax.scatter(plot_df.loc[is_up, "log2FoldChange"],
           plot_df.loc[is_up, "neg_log10_padj"],
           s=10, color="#e05c5c", alpha=0.8, label=f"up ({is_up.sum()})")
ax.scatter(plot_df.loc[is_dn, "log2FoldChange"],
           plot_df.loc[is_dn, "neg_log10_padj"],
           s=10, color="#5c88e0", alpha=0.8, label=f"down ({is_dn.sum()})")

# Threshold lines
ax.axhline(-np.log10(0.05), color="black", linewidth=0.8, linestyle="--", alpha=0.6)
ax.axvline(1,  color="black", linewidth=0.8, linestyle="--", alpha=0.4)
ax.axvline(-1, color="black", linewidth=0.8, linestyle="--", alpha=0.4)

# Label top 5 genes by |LFC|
for gene, row in top.head(5).iterrows():
    lfc = row["log2FoldChange"]
    y   = -np.log10(row["padj"])
    ax.annotate(gene, (lfc, y), fontsize=7,
                xytext=(4, 2), textcoords="offset points")

ax.set_xlabel("log₂ fold change (dex vs untrt)", fontsize=12)
ax.set_ylabel("-log₁₀ adjusted p-value", fontsize=12)
ax.set_title("Volcano plot — airway dataset (DESeq2 via rosetta-bioc)", fontsize=13)
ax.legend(frameon=False, fontsize=10)
plt.tight_layout()
plt.savefig("volcano_airway.png", dpi=150)
plt.show()
print("Saved: volcano_airway.png")

## What just happened

| Step | Tool |
|------|------|
| Load RNA-seq counts | Bioconductor `airway` package (via rpy2) |
| Fit negative binomial model + Wald test | DESeq2 1.52.0 (R 4.6.1, Bioc 3.23) |
| Results back in Python | `rosetta.quick_deseq2()` → `RosettaDataFrame` |
| Volcano plot | matplotlib |

You wrote Python throughout. DESeq2 never left R — rosetta bridged the two transparently.

### Next steps

- **LFC shrinkage:** `model = rb.DESeq2(counts, metadata, '~ dex'); model.run_deseq(); model.lfc_shrink(coef=2, type='apeglm')`
- **edgeR instead:** swap `rb.quick_deseq2` for `rb.quick_edger` — same API
- **sklearn pipeline:** `rb.sklearn_compat` wraps rosetta wrappers as sklearn transformers